# Teoria de Feature Engineering _(version detallada)_

_Seleccion, imputacion, encoding, transformaciones, escalado, PCA y embeddings — con comentarios linea a linea._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering

> 📖 **Version detallada.** Todos los bloques de codigo incluyen comentarios explicando
> el proposito de cada linea y los parametros clave. Para la teoria completa de cada
> seccion consulta `01_feature_engineering_theory.ipynb`.

## Introduccion

El **feature engineering** es el arte de convertir datos crudos en entradas
numericas (las "features") con las que un modelo de machine learning puede
aprender bien. Una buena analogia: el modelo es un cocinero y las features son
los ingredientes ya picados y medidos. Por muy buen cocinero que sea, si le das
ingredientes en mal estado el plato no saldra bien.

Un modelo solo es tan bueno como la **representacion** que recibe. Las
transformaciones correctas logran tres cosas:

- hacen que los patrones sean *separables* (lineal o localmente),
- estabilizan la optimizacion (que el entrenamiento converja),
- y codifican conocimiento del dominio que el modelo, de otro modo, tendria que
  inferir desde cero.

En este notebook recorremos el flujo tipico de feature engineering. Para cada
tecnica vemos primero **(a)** la intuicion y luego **(b)** un ejemplo ejecutable
sobre el dataset real de **Ames Housing** (Kaggle "House Prices"). El objetivo es
**predecir `SalePrice`**, asi que se trata de un problema de **regresion**:

1. **Seleccion de variables** (filter, wrapper, embedded) — *que columnas conservar*
2. **Imputacion** de valores faltantes
3. **Encoding** de categoricas (label / ordinal, one-hot, feature hashing)
4. **Transformaciones** (log, Box-Cox / Yeo-Johnson, binning)
5. **Escalado** (z-score, robusto, min-max, L2)
6. **Reduccion de dimensionalidad** (PCA)
7. **Embeddings** (vectores densos aprendidos)

Cerramos con una **tabla resumen** de cuando usar cada tecnica. Pero antes de
todo eso dedicamos una seccion a la **fuga de datos (data leakage)**: la
disciplina que enmarca *todas* las tecnicas siguientes.

> Una distincion util desde el principio: la **seleccion** de variables *elige un
> subconjunto de las columnas originales* (las mantiene interpretables), mientras
> que la **extraccion** de variables (como PCA) *crea nuevas columnas* combinando
> las originales. Ambas reducen dimensionalidad, pero solo la seleccion conserva
> el significado de cada feature.

### El pipeline de feature engineering de un vistazo

Antes de entrar en cada tecnica, este es el flujo general que recorren los datos:
de **datos crudos** a una **matriz de features** lista para el modelo. El diagrama
de abajo se dibuja con matplotlib (se genera al ejecutar la celda siguiente).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Creamos una figura ancha y baja para el diagrama de pipeline.
# figsize=(12, 2.6): 12 pulgadas de ancho, 2.6 de alto (diagrama horizontal).
fig, ax = plt.subplots(figsize=(12, 2.6))
ax.set_xlim(0, 12); ax.set_ylim(0, 3); ax.axis("off")  # sin ejes visibles

# Lista de etapas del pipeline con su color de fondo.
etapas = [
    ("Datos\ncrudos",                               "#cfe8ff"),
    ("Imputacion +\nseleccion",                     "#cfe8ff"),
    ("Transformaciones\n(encoding, scaling,\nbinning, PCA...)", "#ffe2b3"),
    ("Matriz de\nfeatures",                         "#d6f5d6"),
    ("Modelo de\nML",                               "#f3d1ff"),
]

# Dimensiones de cada bloque: w=ancho, h=alto, gap=espacio entre bloques.
w, h, gap = 2.0, 1.4, 0.35
x = 0.25
centros = []   # guarda el centro x de cada bloque para dibujar las flechas

for texto, color in etapas:
    # FancyBboxPatch: rectangulo con bordes redondeados.
    box = FancyBboxPatch((x, 0.8), w, h, boxstyle="round,pad=0.05",
                         fc=color, ec="#444", lw=1.3)
    ax.add_patch(box)
    ax.text(x + w / 2, 0.8 + h / 2, texto, ha="center", va="center", fontsize=10)
    centros.append(x + w / 2)
    x += w + gap   # avanzamos al siguiente bloque

# Flechas entre bloques consecutivos.
for i in range(len(centros) - 1):
    # FancyArrowPatch: del borde derecho de un bloque al borde izquierdo del siguiente.
    ax.add_patch(FancyArrowPatch(
        (centros[i] + w / 2, 1.5),
        (centros[i + 1] - w / 2, 1.5),
        arrowstyle="-|>", mutation_scale=18, color="#333", lw=1.5,
    ))

ax.text(6, 2.7, "Pipeline de feature engineering", ha="center",
        fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## Setup y datos

Usamos el dataset **Ames Housing** (Kaggle "House Prices", `housing_train.csv`):
1460 casas con 80 columnas y el objetivo `SalePrice`. El helper de abajo localiza
el CSV buscando varias rutas candidatas (y la variable de entorno `HOUSING_CSV`);
si no lo encuentra, cae a un pequeno dataframe sintetico con las columnas clave,
asi el notebook corre igual sin conexion.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

np.random.seed(0)


def load_housing():
    """Localiza housing_train.csv; si no existe, genera datos sinteticos."""
    candidates = []
    if os.environ.get("HOUSING_CSV"):
        candidates.append(os.environ["HOUSING_CSV"])
    candidates += [
        os.path.join("..", "..", "data", "housing_train.csv"),
        os.path.join("..", "..", "..", "data", "housing_train.csv"),
        os.path.join("data", "housing_train.csv"),
    ]
    for path in candidates:
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Cargado: {path}  shape={df.shape}")
            return df
    print("CSV no encontrado — datos sinteticos.")
    rng = np.random.default_rng(0)
    n = 500
    oq  = rng.integers(3, 10, n)
    gla = (oq * 180 + rng.normal(400, 250, n)).clip(400, 5500)
    gla[-4:] = [4600, 5100, 4900, 4700]
    sale = (oq * 22000 + gla * 55 + rng.normal(0, 25000, n)).clip(40000, 600000)
    return pd.DataFrame({
        "Id": np.arange(1, n+1), "OverallQual": oq,
        "GrLivArea": gla.round().astype(int),
        "TotalBsmtSF": (gla*0.6+rng.normal(0,150,n)).clip(0,3000).round().astype(int),
        "1stFlrSF": (gla*0.55+rng.normal(0,100,n)).clip(400,2000).round().astype(int),
        "LotArea": rng.gamma(2.0,5000,n).clip(1300,60000).round().astype(int),
        "LotFrontage": np.where(rng.random(n)<0.18, np.nan, rng.integers(30,120,n)).astype(float),
        "YearBuilt": rng.integers(1900,2010,n),
        "FullBath": rng.integers(1,4,n),
        "GarageCars": rng.integers(0,4,n),
        "MSZoning": rng.choice(["RL","RM","FV","RH"],n),
        "BldgType": rng.choice(["1Fam","2fmCon","Duplex","TwnhsE","Twnhs"],n),
        "Foundation": rng.choice(["PConc","CBlock","BrkTil","Wood","Slab","Stone"],n),
        "CentralAir": rng.choice(["Y","N"],n),
        "Electrical": np.where(rng.random(n)<0.001,np.nan,rng.choice(["SBrkr","FuseA"],n)),
        "ExterQual": rng.choice(["TA","Gd","Ex","Fa"],n),
        "KitchenQual": rng.choice(["TA","Gd","Ex","Fa"],n),
        "BsmtQual": np.where(rng.random(n)<0.03,np.nan,rng.choice(["TA","Gd","Ex","Fa"],n)),
        "FireplaceQu": np.where(rng.random(n)<0.47,np.nan,rng.choice(["TA","Gd","Ex","Fa","Po"],n)),
        "PoolQC": np.where(rng.random(n)<0.995,np.nan,rng.choice(["Ex","Gd"],n)),
        "Alley": np.where(rng.random(n)<0.94,np.nan,rng.choice(["Grvl","Pave"],n)),
        "GarageType": np.where(rng.random(n)<0.06,np.nan,rng.choice(["Attchd","Detchd","BuiltIn"],n)),
        "HeatingQC": rng.choice(["Ex","Gd","TA","Fa","Po"],n),
        "Neighborhood": rng.choice(["NAmes","CollgCr","OldTown","Edwards","Gilbert",
                                    "NridgHt","Sawyer","NWAmes","SawyerW","Mitchel",
                                    "BrkSide","Crawfor","IDOTRR","Timber","NoRidge",
                                    "StoneBr","SWISU","ClearCr","MeadowV","Blmngtn",
                                    "BrDale","Veenker","NPkVill","Blueste","Greens"],n),
        "SalePrice": sale.round().astype(int),
    })


df = load_housing()
print(f"shape={df.shape}")


In [ ]:
# Un primer vistazo a tipos y valores faltantes.
# Las decisiones de feature engineering (que imputar, como codificar)
# dependen directamente de cuantos NaN hay y en que columnas.
print("shape:", df.shape)
print(f"Columnas: {df.shape[1]}  |  Filas: {df.shape[0]}")

# Contamos los NaN por columna y los ordenamos de mayor a menor.
na = df.isna().sum()
na_pct = (na / len(df) * 100).round(1)

print("\nColumnas con valores faltantes (ordenadas por cantidad):")
miss = pd.DataFrame({"n_missing": na, "pct_missing": na_pct})
miss = miss[miss["n_missing"] > 0].sort_values("n_missing", ascending=False)
print(miss.to_string())

# Interpretacion clave para Ames:
# - Muchos NaN en PoolQC, Alley, FireplaceQu, GarageType = "feature ausente" (la casa no tiene eso).
# - Pocos NaN en LotFrontage = dato genuinamente perdido.
# Esta distincion determina la estrategia de imputacion (seccion 2).


## 0. Fuga de datos (data leakage) — la regla de oro

**Regla:** separa primero (`train_test_split`), ajusta solo en train (`fit`),
aplica en train y test (`transform`). Nunca uses datos de test en el `fit`.

`Pipeline` + `ColumnTransformer` aplican esta regla automaticamente.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Demo numerica: mostramos el efecto de la CONTAMINACION train/test.
# Tomamos GrLivArea, separamos train/test, y vemos que media aprende
# un StandardScaler si hace fit sobre TODO vs solo en train.
g = df[["GrLivArea"]].to_numpy(dtype=float)

# Separamos 70% train / 30% test. El split debe ocurrir ANTES de cualquier fit.
g_train, g_test = train_test_split(g, test_size=0.3, random_state=0)

# (MAL) fit sobre TODO el dataset antes del split: el scaler "mira" filas de test.
# La media aprendida incluye informacion del test -> fuga de datos.
scaler_leak = StandardScaler().fit(g)             # fit con train + test juntos
mean_leak   = scaler_leak.mean_[0]               # media que usara para escalar

# (BIEN) fit SOLO en train. El test se transforma despues usando los parametros de train.
scaler_ok = StandardScaler().fit(g_train)         # fit solo con train
mean_ok   = scaler_ok.mean_[0]                   # media calculada sin ver el test
g_test_ok = scaler_ok.transform(g_test)           # aplica parametros de train al test

print(f"Media aprendida — fit con TODO (con fuga)  : {mean_leak:.2f} sqft")
print(f"Media aprendida — fit solo train (correcto): {mean_ok:.2f} sqft")
print(f"Diferencia: {abs(mean_leak - mean_ok):.2f} sqft")
print()
print("Con 500 filas la diferencia es pequena, pero con pocos datos o alta varianza")
print("puede ser significativa y sesgar la evaluacion del modelo.")


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge

# Columnas para la demo: numericas y categoricas.
num_d = ["GrLivArea", "LotArea", "OverallQual"]
cat_d = ["MSZoning", "CentralAir"]
Xd = df[num_d + cat_d].dropna(subset=cat_d)
yd = df.loc[Xd.index, "SalePrice"]

# Pipeline numerico: imputa mediana -> estandariza.
# fit aprende mediana y media/std SOLO de train -> sin leakage.
num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc",  StandardScaler())])

# Pipeline categorico: imputa moda -> one-hot.
# handle_unknown="ignore": categorias nuevas en test -> vector de ceros (sin error).
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

# ColumnTransformer: enruta cada grupo de columnas al pipeline correcto.
pre = ColumnTransformer([("num", num_pipe, num_d), ("cat", cat_pipe, cat_d)])

# Pipeline final: preprocesamiento + Ridge.
# cross_val_score re-ajusta el pipeline en cada fold SOLO con datos de train.
pipe = Pipeline([("pre", pre), ("ridge", Ridge(alpha=1.0))])
scores = cross_val_score(pipe, Xd, yd, cv=5, scoring="neg_root_mean_squared_error")
print(f"RMSE por fold: {(-scores).round(0)}")
print(f"RMSE medio: {(-scores).mean():,.0f}  +/- {(-scores).std():,.0f}")


## 1. Seleccion de variables

- **Filter** (VarianceThreshold, f_regression, mutual_info_regression, SelectKBest): puntua cada feature independientemente.
- **Wrapper** (RFE): entrena un modelo eliminando features iterativamente.
- **Embedded** (Lasso, feature_importances_): la seleccion ocurre dentro del entrenamiento.

In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_regression, f_regression, SelectKBest, RFE)

# Bloque numerico para los metodos de seleccion.
num_features = [c for c in ["OverallQual","GrLivArea","GarageCars","TotalBsmtSF",
                             "1stFlrSF","FullBath","YearBuilt","LotArea"] if c in df.columns]
sel = df[num_features].fillna(df[num_features].median(numeric_only=True))
y_sel = df["SalePrice"].fillna(df["SalePrice"].median()).astype(float)
idx = sel.index.intersection(y_sel.dropna().index)
sel = sel.loc[idx]; y_sel = y_sel.loc[idx]

# (a) VarianceThreshold: descarta columnas casi constantes (varianza <= umbral).
# Aqui lo usamos para VER la varianza, no para filtrar.
vt = VarianceThreshold(threshold=0.0).fit(sel)
print("Varianza por columna:", dict(zip(sel.columns, np.round(vt.variances_, 0))))

# (b) Correlacion de Pearson (solo captura relacion LINEAL).
corr = sel.apply(lambda c: np.corrcoef(c, y_sel)[0, 1]).abs().sort_values(ascending=False)
print("\n|Pearson| con SalePrice:")
print(corr.round(3).to_string())

# (c) f_regression: estadistico F para relacion lineal entre cada feature y SalePrice.
# mutual_info_regression: captura dependencias NO lineales.
F, p = f_regression(sel, y_sel)
mi = mutual_info_regression(sel, y_sel, random_state=0)
comparison = pd.DataFrame({"F": F, "p": p, "MI": mi}, index=sel.columns).sort_values("F", ascending=False)
print("\nf_regression vs mutual_info_regression:")
print(comparison.round(3))

# SelectKBest: selecciona las k features con mayor score.
kbest = SelectKBest(score_func=f_regression, k=4).fit(sel, y_sel)
print("\nTop-4 por F:", list(sel.columns[kbest.get_support()]))


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor

# Estandarizamos antes de RFE: los coeficientes deben ser comparables entre features.
sel_std = StandardScaler().fit_transform(sel)

# RFE (Wrapper): entrena LinearRegression, elimina la feature de menor coeficiente,
# repite hasta quedar con n_features_to_select=4.
rfe = RFE(LinearRegression(), n_features_to_select=4).fit(sel_std, y_sel)
print("RFE seleccionadas:", list(sel.columns[rfe.support_]))
print("RFE ranking (1=elegida):", dict(zip(sel.columns, rfe.ranking_)))

# Lasso (Embedded): alpha controla la penalizacion L1.
# Mayor alpha -> mas coeficientes en exactamente CERO -> seleccion mas agresiva.
lasso = Lasso(alpha=0.1).fit(StandardScaler().fit_transform(sel), y_sel)
lasso_coef = pd.Series(lasso.coef_, index=sel.columns).sort_values(key=abs, ascending=False)
print("\nLasso coeficientes (0 = descartada):"); print(lasso_coef.round(2).to_string())

# RandomForest (Embedded): feature_importances_ = fraccion de varianza reducida
# por cada feature en todos los arboles. Suma 1.
rf = RandomForestRegressor(n_estimators=200, random_state=0).fit(sel, y_sel)
imp = pd.Series(rf.feature_importances_, index=sel.columns).sort_values(ascending=False)
print("\nImportancia RF:"); print(imp.round(3).to_string())


## 2. Imputacion de valores faltantes

En Ames, la mayoria de NaN significan **"feature ausente"** (no dato perdido):
`PoolQC` NaN = sin piscina -> rellenar con `'None'`/`0`. Solo algunos son genuinamente
perdidos: `LotFrontage` -> mediana; `Electrical` -> moda.

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

cols_check = [c for c in ["PoolQC","Alley","FireplaceQu","GarageType","LotFrontage","Electrical"] if c in df.columns]
print("Faltantes:"); print(df[cols_check].isna().sum().to_string())

# (a) NaN = 'feature ausente' -> constante 'None'.
# fill_value="None" marca que la casa no tiene esa caracteristica (sin piscina, etc.).
if "PoolQC" in df.columns:
    imp_none = SimpleImputer(strategy="constant", fill_value="None")
    poolqc_imp = imp_none.fit_transform(df[["PoolQC"]])
    print(f"\nPoolQC unicos tras 'None': {pd.unique(poolqc_imp.ravel())[:5]}")

# (b) Faltante genuino numerico -> mediana (robusta a outliers).
# statistics_[0]: el valor aprendido en fit (mediana del set de entrenamiento).
if "LotFrontage" in df.columns:
    imp_med = SimpleImputer(strategy="median")
    lf = imp_med.fit_transform(df[["LotFrontage"]])
    print(f"LotFrontage mediana aprendida: {imp_med.statistics_[0]:.1f} | NaN restantes: {np.isnan(lf).sum()}")

# (c) add_indicator=True: agrega columna binaria 1=faltaba/0=estaba.
# Util cuando la ausencia misma es informativa (casa sin garaje -> precio diferente).
if "LotFrontage" in df.columns:
    imp_ind = SimpleImputer(strategy="median", add_indicator=True)
    lf_flag = imp_ind.fit_transform(df[["LotFrontage"]])
    print(f"Con add_indicator: {lf_flag.shape[1]} columnas (valor + flag). Faltantes marcados: {int(lf_flag[:,1].sum())}")

# (d) KNNImputer: rellena con el promedio de los k vecinos mas parecidos.
# Requiere escalar primero porque usa distancias euclidianas.
knn_cols = [c for c in ["LotFrontage","LotArea","GrLivArea","OverallQual","YearBuilt"] if c in df.columns]
block = df[knn_cols].copy()
sc_knn = StandardScaler()
block_std = sc_knn.fit_transform(block)              # escala preservando NaN
knn = KNNImputer(n_neighbors=5)
block_imp = sc_knn.inverse_transform(knn.fit_transform(block_std))  # imputa y vuelve a escala original
print(f"\nKNNImputer: NaN restantes = {np.isnan(block_imp).sum()}")


## 3. Encoding de categoricas

- **Ordinal**: para variables con orden conocido (Po<Fa<TA<Gd<Ex). Impone el orden correcto.
- **One-hot**: para nominales (sin orden). Cada categoria -> columna indicadora 0/1.
- **Feature hashing**: cardinalidad alta o desconocida. Mapeo a `m` buckets fijos via hash.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Escala de calidad ORDINAL real de Ames: None=0 (no existe), Po=1, ..., Ex=5.
# Imponemos el orden con categories=[...] para que el encoder use el orden SEMANTICO,
# no el alfabetico (que seria incorrecto: Ex < Fa < Gd < ... ).
quality_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
qual_cols = [c for c in ["ExterQual","KitchenQual","HeatingQC","FireplaceQu","BsmtQual"] if c in df.columns]
qwork = df[qual_cols].fillna("None")   # NaN = la casa no tiene esa parte
oe = OrdinalEncoder(categories=[quality_order]*len(qual_cols))
qual_codes = oe.fit_transform(qwork)
print("Orden:", dict(zip(quality_order, range(len(quality_order)))))
if "KitchenQual" in qual_cols:
    ki = qual_cols.index("KitchenQual")
    sample = list(zip(df["KitchenQual"].head(6), qual_codes[:6,ki].astype(int)))
    print("KitchenQual -> codigo:", sample)


In [ ]:
# One-hot para variables NOMINALES (MSZoning, BldgType, Foundation).
# Todas las categorias quedan a la misma distancia (no hay orden implicito).
cat_cols_oh = [c for c in ["MSZoning","BldgType","Foundation","CentralAir"] if c in df.columns]
work = df.dropna(subset=cat_cols_oh).copy()

# pd.get_dummies: version rapida para EDA. drop_first=False: conserva todas las columnas.
dummies = pd.get_dummies(work[cat_cols_oh], drop_first=False)
print("Columnas get_dummies:", list(dummies.columns))

# OneHotEncoder sklearn: version de produccion (se ajusta solo en train).
# handle_unknown="ignore": categoria nueva en test -> fila de ceros (sin error).
# sparse_output=True: devuelve matriz dispersa (eficiente con muchos ceros).
enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_oh = enc.fit_transform(work[cat_cols_oh])
print(f"One-hot sklearn: shape={X_oh.shape}  densidad={X_oh.nnz/(X_oh.shape[0]*X_oh.shape[1]):.3f}")


In [ ]:
from sklearn.feature_extraction import FeatureHasher

# Feature hashing: mapea cualquier numero de categorias a m buckets fijos.
# No necesita fit (sin estado): no hay vocabulario que guardar.
# Util cuando la cardinalidad es alta o desconocida en advance.
n_buckets = 8
hasher = FeatureHasher(n_features=n_buckets, input_type="string")

# Transformamos el Neighborhood (25 barrios) a 8 buckets fijos.
# Cada valor se convierte en una lista de tokens ("nbhd=NAmes") para el hasher.
tokens = [[f"nbhd={v}"] for v in df["Neighborhood"].astype(str)]
H = hasher.transform(tokens)   # sin fit: es sin estado
print(f"{df['Neighborhood'].nunique()} barrios -> {n_buckets} buckets  shape={H.shape}")

# Colisiones: con m<cardinalidad, varios barrios pueden caer en el mismo bucket.
# La probabilidad de colision por par es ~1/m; el hash de signo (+/-1) las mitiga.
print("\nPrimeras 4 filas hasheadas:")
print(pd.DataFrame(H.toarray()[:4], index=df["Neighborhood"].head(4).values).round(0))


In [ ]:
# DEMO DE COLISIONES: como el numero de buckets m afecta la tasa de colision.
# Una colision ocurre cuando dos categorias distintas se mapean al mismo bucket.
# Con m << K (numero de categorias), las colisiones son frecuentes.
# Con m >> K, son raras. La recomendacion es usar m = potencia de 2 >= K.
def bucket_of(value, m):
    """Devuelve el indice del bucket al que cae 'value' con m buckets."""
    h = FeatureHasher(n_features=m, input_type="string")
    row = h.transform([[f"nbhd={value}"]]).toarray()[0]
    # flatnonzero: indices donde la fila tiene valor != 0 (el bucket activo).
    nonzero = row.nonzero()[0]
    return int(nonzero[0]) if len(nonzero) > 0 else None

barrios = df["Neighborhood"].dropna().unique()
K = len(barrios)  # cardinalidad real (numero de barrios distintos)
print(f"Cardinalidad real (K): {K} barrios")
print()

# Comparamos la tasa de colision para distintos m.
for m in (128, 64, 16, 8):
    # Para cada barrio, calculamos a que bucket cae.
    mapping = {b: bucket_of(b, m) for b in barrios}
    # Numero de colisiones = barrios - buckets distintos usados.
    n_collide = K - len(set(mapping.values()))
    print(f"  m={m:>4} buckets | colisiones: {n_collide:>2} de {K} barrios "
          f"({n_collide/K:.0%})", end="")
    print("  <- m >= K, pocas colisiones" if m >= K else "  <- m < K, mas colisiones")

# Conclusion: con m=64 >= K=25 barrios las colisiones son raras.
# Con m=8 <<  K=25, muchos barrios comparten el mismo bucket -> ruido.


## 4. Transformaciones (log, Box-Cox, binning)

`SalePrice` y `LotArea` tienen sesgo derecho (skewness >> 0). `log1p` comprime colas;
`PowerTransformer` estima el exponente optimo. `KBinsDiscretizer` convierte continuas en cajones.

In [ ]:
from sklearn.preprocessing import PowerTransformer, KBinsDiscretizer

def skewness(a):
    a = np.asarray(a).ravel(); m = a.mean(); s = a.std()
    return float(((a-m)**3).mean()/s**3) if s>0 else 0.0

price  = df["SalePrice"].to_numpy(dtype=float)
lot    = df["LotArea"].to_numpy(dtype=float).reshape(-1,1)

# log1p: comprime colas derechas sin eliminar puntos.
# x' = log(1+x); inversa: expm1(x') = x. Maneja x=0 sin error.
price_log = np.log1p(price)
lot_log   = np.log1p(lot)

# Box-Cox: estima el lambda optimo para normalizar (exige x>0).
# Yeo-Johnson: generaliza Box-Cox admitiendo 0 y negativos.
# standardize=True (default): la salida queda ademas con media=0 y varianza=1.
lot_bc = PowerTransformer(method="box-cox").fit_transform(lot)     # solo x>0
lot_yj = PowerTransformer(method="yeo-johnson").fit_transform(lot)  # x>=0

print(f"SalePrice: skew original={skewness(price):+.3f}  log1p={skewness(price_log):+.3f}")
print(f"LotArea:   skew original={skewness(lot):+.3f}  log1p={skewness(lot_log):+.3f}  Box-Cox={skewness(lot_bc):+.3f}  YJ={skewness(lot_yj):+.3f}")

# KBinsDiscretizer: convierte continua en cajones ordinales.
# strategy='uniform': igual ancho | 'quantile': igual frecuencia (robusto al sesgo).
gr = df[["GrLivArea"]]
kb_u = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform").fit_transform(gr)
kb_q = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile").fit_transform(gr)
print(f"\nKBinsDiscretizer uniform  -> bins: {np.unique(kb_u)}")
print(f"KBinsDiscretizer quantile -> bins: {np.unique(kb_q)}")


In [ ]:
# Visualizacion: distribucion ANTES y DESPUES de log1p y Box-Cox.
# El objetivo es ver visualmente como cada transformacion reduce el sesgo.
# Una distribucion mas simetrica (cercana a la normal) ayuda a los modelos lineales.
from sklearn.preprocessing import PowerTransformer

def skewness(a):
    a = np.asarray(a).ravel(); m = a.mean(); s = a.std()
    return float(((a - m)**3).mean() / s**3) if s > 0 else 0.0

price = df["SalePrice"].to_numpy(dtype=float)
price_log  = np.log1p(price)
price_bc   = PowerTransformer(method="yeo-johnson").fit_transform(price.reshape(-1, 1)).ravel()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# Panel 1: SalePrice original (sesgo a la derecha).
axes[0].hist(price, bins=40, color="#4c78a8", edgecolor="white")
axes[0].set_title(f"SalePrice original\nskewness = {skewness(price):+.2f}")
axes[0].set_xlabel("USD"); axes[0].set_ylabel("conteo")

# Panel 2: log1p(SalePrice) — comprime la cola derecha.
axes[1].hist(price_log, bins=40, color="#54a24b", edgecolor="white")
axes[1].set_title(f"log1p(SalePrice)\nskewness = {skewness(price_log):+.2f}")
axes[1].set_xlabel("log1p(USD)")

# Panel 3: Yeo-Johnson — estima la transformacion optima para normalizar.
axes[2].hist(price_bc, bins=40, color="#e45756", edgecolor="white")
axes[2].set_title(f"Yeo-Johnson(SalePrice)\nskewness = {skewness(price_bc):+.2f}")
axes[2].set_xlabel("transformado (estandarizado)")

plt.suptitle("Efecto de las transformaciones sobre el sesgo de SalePrice",
             fontweight="bold", y=1.03)
plt.tight_layout(); plt.show()
print("Entre mas cercano a 0 el skewness, mas simetrica (y parecida a la normal) es la distribucion.")


In [ ]:
# Binning manual con pd.cut: convertimos YearBuilt en epocas de construccion.
# Por que binear YearBuilt?
#   - La relacion precio/ano no es lineal: casas de los 50s valen diferente
#     a las de los 2000s, pero la diferencia no crece/decrece uniformemente.
#   - Los modelos lineales no capturan efectos de umbral ("antes/despues de 1970").
#   - Un modelo de arboles PODRIA aprenderlo, pero los bins lo hacen explicito.
bins_yr  = [0, 1940, 1960, 1980, 2000, 2010]
labels_yr = ["pre-1940", "1940s-50s", "1960s-70s", "1980s-90s", "2000s"]

# pd.cut: asigna cada valor de YearBuilt al bin correspondiente.
# bins: limites (intervals abiertos a la izquierda, cerrados a la derecha).
# labels: nombre de cada bin (en el mismo orden que los intervalos).
# right=True (default): el limite derecho de cada bin SI pertenece al bin.
df["YearBuilt_era"] = pd.cut(df["YearBuilt"], bins=bins_yr, labels=labels_yr, right=True)

print("Distribucion de casas por epoca de construccion:")
era_counts = df["YearBuilt_era"].value_counts().sort_index()
print(era_counts)
print()

# Precio medio por epoca: muestra que la era es informativa.
if "SalePrice" in df.columns:
    precio_por_era = df.groupby("YearBuilt_era", observed=True)["SalePrice"].agg(
        n="count", media="mean", mediana="median"
    ).round(0)
    print("Precio segun la epoca de construccion:")
    print(precio_por_era)

# Limpieza: eliminamos la columna auxiliar para no alterar el resto del notebook.
df.drop(columns=["YearBuilt_era"], inplace=True)


## 5. Escalado

| Scaler | Formula | Robusto a outliers | Uso |
|---|---|---|---|
| StandardScaler | $(x-\mu)/\sigma$ | No | SVM, k-NN, PCA, lineal+reg |
| RobustScaler | $(x-\text{med})/(Q_3-Q_1)$ | Si | Features con outliers |
| MinMaxScaler | $(x-x_{\min})/(x_{\max}-x_{\min})$ | No | Entradas acotadas |
| Normalizer | $x/\|x\|_2$ (por fila) | — | Similitud coseno, TF-IDF |

In [ ]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler, Normalizer

num_sc = df[["GrLivArea","LotArea"]].dropna()

# StandardScaler: fit aprende media y std de train.
# SENSIBLE a outliers: un solo valor extremo infla std y comprime el resto.
sc_std  = StandardScaler().fit(num_sc)
z_std   = sc_std.transform(num_sc)
print("StandardScaler -> media aprendida (GrLivArea):", sc_std.mean_[0].round(1))
print(f"  tras escalar: media={z_std[:,0].mean():.3f}  std={z_std[:,0].std():.3f}")

# RobustScaler: fit aprende mediana y IQR (robustos a outliers).
# Formula: (x - mediana) / (Q3-Q1). No se deja arrastrar por valores extremos.
sc_rob  = RobustScaler().fit(num_sc)
z_rob   = sc_rob.transform(num_sc)
print(f"RobustScaler -> center (mediana): {sc_rob.center_[0]:.1f}  scale (IQR): {sc_rob.scale_[0]:.1f}")

# MinMaxScaler: mapea cada columna a [0,1]. SENSIBLE a outliers: un extremo estira el rango.
sc_mm   = MinMaxScaler().fit(num_sc)
z_mm    = sc_mm.transform(num_sc)
print(f"MinMaxScaler -> min={z_mm.min(axis=0).round(3)}  max={z_mm.max(axis=0).round(3)}")

# Normalizer: normaliza cada FILA (muestra) a norma L2 = 1.
# Actua por fila, no por columna (a diferencia de los escaladores anteriores).
l2 = Normalizer(norm="l2")
z_l2 = l2.fit_transform(num_sc.values[:5])
print(f"Normalizer L2 -> normas de fila: {np.linalg.norm(z_l2, axis=1).round(3)}")


In [ ]:
# Visualizacion comparativa: como cada scaler TRATA los outliers de GrLivArea.
# Insertamos manualmente 3 outliers extremos para que la diferencia sea visible.
area = df["GrLivArea"].dropna().to_numpy(dtype=float).reshape(-1, 1)
# area_out: dataset con outliers inyectados (similar a las casas grandes de Ames).
area_out = np.vstack([area, [[4800], [5200], [5500]]])

# Ajustamos los tres escaladores sobre el dataset con outliers.
z_std = StandardScaler().fit_transform(area_out)   # usa media y std -> sensible
z_rob = RobustScaler().fit_transform(area_out)      # usa mediana e IQR -> robusto
z_mm  = MinMaxScaler().fit_transform(area_out)      # estira a [0,1] -> sensible

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
kw = dict(bins=35, edgecolor="white", alpha=0.85)

# Panel 1: StandardScaler — media=0 y std=1. Los outliers inflan la std,
# lo que comprime los valores normales hacia 0.
axes[0].hist(z_std, color="#4c78a8", **kw)
axes[0].axvline(0, c="red", lw=2, ls="--", label="media=0")
axes[0].set_title("StandardScaler\n(media y std — sensible a outliers)")
axes[0].set_xlabel("z-score"); axes[0].legend(fontsize=8)

# Panel 2: RobustScaler — centra con la mediana, escala con IQR.
# Los outliers no afectan mediana ni IQR, asi que los valores normales
# quedan bien distribuidos aunque los outliers queden lejos.
axes[1].hist(z_rob, color="#54a24b", **kw)
axes[1].axvline(0, c="red", lw=2, ls="--", label="mediana=0")
axes[1].set_title("RobustScaler\n(mediana e IQR — robusto a outliers)")
axes[1].set_xlabel("(x - mediana) / IQR"); axes[1].legend(fontsize=8)

# Panel 3: MinMaxScaler — rango [0,1]. Un unico outlier extremo
# estira todo el rango y comprime los valores normales hacia 0.
axes[2].hist(z_mm, color="#e45756", **kw)
axes[2].axvline(1, c="red", lw=2, ls="--", label="max=1")
axes[2].set_title("MinMaxScaler\n(rango [0,1] — muy sensible a outliers)")
axes[2].set_xlabel("[0, 1]"); axes[2].legend(fontsize=8)

plt.suptitle("Impacto de los outliers en cada scaler (GrLivArea + 3 outliers inyectados)",
             fontweight="bold", y=1.03)
plt.tight_layout(); plt.show()

print("Lectura: con outliers extremos, StandardScaler y MinMaxScaler comprimen")
print("todos los valores normales en un rango estrecho. RobustScaler los preserva.")


## 6. Reduccion de dimensionalidad — PCA

PCA encuentra las direcciones (componentes principales) que maximizan la varianza.
**Siempre estandarizar antes**: sin escalar, la feature de mayor rango domina.
`explained_variance_ratio_`: fraccion de varianza por componente.

In [ ]:
from sklearn.decomposition import PCA

pca_cols = [c for c in ["OverallQual","GrLivArea","GarageCars","TotalBsmtSF",
                         "1stFlrSF","FullBath","YearBuilt"] if c in df.columns]
Xp = df[pca_cols].dropna()

# PASO 1: Estandarizar. Sin esto, GrLivArea (en cientos) domina todas las componentes.
Xp_std = StandardScaler().fit_transform(Xp)

# PCA(): sin n_components -> calcula todas las componentes.
# fit aprende los autovectores (components_) y autovalores (explained_variance_).
pca = PCA().fit(Xp_std)
evr = pca.explained_variance_ratio_
print("Varianza explicada por componente:", evr.round(3))
print("Varianza acumulada:", np.cumsum(evr).round(3))

# n_components=0.95: retiene las componentes necesarias para explicar el 95% de varianza.
pca95 = PCA(n_components=0.95).fit(Xp_std)
print(f"\nComponentes necesarias para 95% de varianza: {pca95.n_components_}")

# transform: proyecta los datos sobre las primeras 2 componentes para visualizar.
Z = PCA(n_components=2).fit_transform(Xp_std)
target = df.loc[Xp.index, "SalePrice"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(range(1,len(evr)+1), evr, alpha=0.7, label="individual")
ax[0].plot(range(1,len(evr)+1), np.cumsum(evr), "o-r", label="acumulada")
ax[0].axhline(0.9, ls="--", c="gray"); ax[0].set_title("Scree plot"); ax[0].legend()
sc = ax[1].scatter(Z[:,0], Z[:,1], c=target, cmap="viridis", s=12, alpha=0.6)
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
ax[1].set_title("Proyeccion PC1/PC2 coloreada por SalePrice")
plt.colorbar(sc, ax=ax[1], label="SalePrice")
plt.tight_layout(); plt.show()


## 7. Embeddings — vectores densos aprendidos

One-hot es disperso ($K$ dimensiones, solo 1 activa). Un **embedding** asigna a cada
categoria un vector denso $\mathbf{e}_c \in \mathbb{R}^d$ con $d \ll K$, aprendido
por descenso de gradiente. Categorias parecidas quedan cercanas en el espacio.

In [ ]:
# Mostramos el contraste estructural entre one-hot y embedding.
K = 25   # numero de barrios en Ames

# One-hot: vector de K=25 dimensiones con un unico 1 (disperso).
onehot = np.zeros(K); onehot[7] = 1

# Embedding: vector de d=3 dimensiones (denso, numeros reales aprendidos).
embed = np.array([0.21, -0.84, 0.57])

fig, ax = plt.subplots(2, 1, figsize=(11, 3.2))
ax[0].imshow(onehot.reshape(1,-1), cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax[0].set_title(f"One-hot: {K}-d, disperso (un solo 1, el resto 0)")
ax[0].set_yticks([]); ax[0].set_xticks(range(0,K,2))
ax[1].imshow(embed.reshape(1,-1), cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
for j,v in enumerate(embed): ax[1].text(j, 0, f"{v:.2f}", ha="center", va="center", fontsize=11)
ax[1].set_title("Embedding: 3-d, denso (numeros reales con significado aprendido)")
ax[1].set_yticks([]); ax[1].set_xticks(range(3))
plt.tight_layout(); plt.show()

# Ventaja de embeddings para alta cardinalidad: memoria y parametros.
for K_demo in [K, 1000, 1_000_000]:
    print(f"K={K_demo:>9,}: one-hot={K_demo:>9,} dims (disperso) | "
          f"embedding(d=8)={K_demo*8:>10,} params (denso)")


In [ ]:
# nn.Embedding (PyTorch): la tabla de embeddings E de forma (K, d).
# Recibe indices enteros (barrio -> indice) y devuelve la fila correspondiente.
# Sus pesos se entrenan por backpropagation junto con el modelo.
nbhd_cat = df["Neighborhood"].astype("category")
vocab = list(nbhd_cat.cat.categories)
idx   = nbhd_cat.cat.codes.to_numpy().astype("int64")
print(f"Vocabulario: {len(vocab)} barrios")

try:
    import torch; import torch.nn as nn
    torch.manual_seed(0)
    embed_dim = 8   # 8-d << 25 categorias
    emb = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim)
    # emb.weight: tabla de embeddings E con forma (K, d), entrenada por gradiente.
    print(f"nn.Embedding: tabla E shape={tuple(emb.weight.shape)} (una fila por barrio)")
    sample = torch.tensor(idx[:5])
    print("Vectores para las 5 primeras casas:")
    print(emb(sample).detach().numpy().round(3))
except Exception as e:
    print(f"torch no disponible ({e}); ilustracion con NumPy:")
    rng_e = np.random.default_rng(0)
    E = rng_e.normal(0,1,(len(vocab),8))
    print(f"Tabla E shape={E.shape}")
    print("Lookup (indice -> fila de E):", E[idx[:5]].round(3))


## Resumen — tabla de referencia rapida

| Tecnica | Que hace | Usar cuando | Cuidado con |
|---|---|---|---|
| Seleccion (filter/wrapper/embedded) | elige columnas originales | reducir ruido/redundancia | filtros ignoran interacciones |
| Imputacion | rellena NaN | modelo no acepta NaN | en Ames, NaN suele ser 'ausente' -> 'None'/0 |
| Ordinal encoding | categoria -> entero con orden | ordinales reales (Po<..<Ex) | impone orden falso en nominales |
| One-hot | categoria -> vector 0/1 | nominales de baja cardinalidad | explota con K alto |
| Feature hashing | categoria -> m buckets via hash | K alto/desconocido | colisiones; no interpretable |
| log1p | comprime colas | datos positivos sesgados | requiere x>-1 |
| Box-Cox / Yeo-Johnson | potencia que normaliza | estabilizar varianza | BC exige x>0 |
| Binning | continua -> cajones | efectos de umbral | n_bins es hiperparametro |
| StandardScaler | media=0, std=1 | SVM, k-NN, PCA, lineal+reg | sensible a outliers |
| RobustScaler | centra con mediana/IQR | features con outliers | no acota a rango fijo |
| MinMaxScaler | rango [0,1] | entradas acotadas | outliers estiran el rango |
| Normalizer L2 | filas de longitud 1 | similitud coseno, TF-IDF | por muestra, no por feature |
| PCA | proyecta maximizando varianza | decorrelacionar, comprimir | estandarizar primero; pierde interpretabilidad |
| Embeddings | vectores densos aprendidos | alta cardinalidad | requiere entrenar; parte del modelo |

**Reglas de oro**

1. **Ajusta solo en train** (`fit` en train, `transform` en train y test).
2. Empareja la transformacion con el modelo: arboles no necesitan escalar; distancias/gradientes si.
3. Usa `Pipeline` + `ColumnTransformer` para que entrenamiento y serving sean identicos.